In [1]:
%load_ext autoreload
%autoreload 2

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

from sklearn.manifold import TSNE, MDS
from torch.utils.data import Subset, DataLoader

import sys
import torch
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")
print(sys.executable)

/home/kathryn/miniconda3/envs/py37/bin/python


In [2]:
from SeqTransformerAE import SeqAE
from TRAVIS_data_padded import VAEDataset

### Check compute 

In [3]:
use_cuda = torch.cuda.is_available()
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))
num_cores = !cat /proc/cpuinfo | grep processor | wc -l
num_cores = num_cores[0]
print(num_cores)

1
TITAN Xp
12


### Dataset

In [4]:
data_path = 'data/partial_data.csv'
property_pred = None
dataset = VAEDataset(data_path, property_pred=property_pred, use_cuda=use_cuda)

In [5]:
model = SeqAE(n_tokens = dataset.n_tokens,
             d_model=256,nhead=16,d_latent=64,num_encoder_layers=3,
             num_decoder_layers=3,dim_feedforward=512,dropout=0.1,
             activation='relu',layer_norm_eps=1e-5,batch_first=True)

n_steps = 200 #200
batch_size = 2
lr = 0.00001

sampler = torch.utils.data.RandomSampler(dataset,replacement = False,
                                         num_samples = n_steps*batch_size)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=8, sampler = sampler)

optimizer = torch.optim.Adam(model.parameters(), lr = lr)
scheduler = None

In [7]:
from TRAVIS_training_functions import fit
loss_out = fit(model, dataloader, optimizer, scheduler,
               n_epochs = 2, n_steps = n_steps, 
               use_cuda = use_cuda, prop_pred_loss = False,
               recon_alpha = 1., variational = False, use_out_mask = False)

Training in progress...:   0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

In [10]:
dataset.__getitem__(0)

{'seq': tensor([12, 27,  5, 27,  1, 20,  6, 16,  1, 13, 21,  2, 20,  7, 16, 16,  1, 20,
          1, 16, 16,  2, 16, 27,  8, 27, 27, 27, 27,  1,  4, 27,  9, 27, 27, 27,
          1, 16,  0, 20,  2, 27, 27,  9,  2, 27,  8,  2, 16, 16,  7, 16,  6, 13,
         21,  2, 27, 27,  1, 16, 28,  2, 27, 27,  5, 16, 28, 14, 24, 24, 24, 24,
         24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
         24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
         24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24]),
 'pad_mask': tensor([False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, Fal

In [16]:
samp = dataset.__getitem__(0)
# print(samp)
model.forward(samp)

AttributeError: 'dict' object has no attribute 'size'

In [29]:
torch.cuda.empty_cache()